# 💊 Pharma Supplier Document Assistant: Full RAG Pipeline + Gradio UI

**Author:** Bashar

An end-to-end, fully open-source RAG chatbot for supplier documentation (no Gemini or other paid API):

| Stage | Implementation |
|---|---|
| Document processing | PyMuPDF for digital pages; **EasyOCR** for scanned pages (detected automatically), with table-row rebuilding and identifier clean-up |
| Metadata tagging | Every page classified and split into documents with Mistral-7B (constrained JSON + rules from Project 7): `doc_type`, `doc_id`, `page_range`, `identifier`, `source_file`, `ocr` |
| Embeddings & index | `BAAI/bge-small-en-v1.5` (sentence-transformers) in a LlamaIndex vector index; 256-token chunks |
| Routing & retrieval | The question is classified into a document type and the search is filtered on it, with fallbacks when the routed search looks wrong; top-4 chunks |
| Smart prompting | Numbered sources; the model must cite `[n]` after every fact, copy values exactly, and say when the answer is not in the documents |
| Open-source LLM | **Mistral-7B-Instruct v0.2** (4-bit GGUF, llama.cpp) |
| Answer + sources | Streamed answer, cited sources, similarity per chunk, chunk count and a **confidence level** |
| UI | Gradio: multi-PDF upload, automatic document library, search scope, chat history, sources table, chat export |

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run all cells; the last one prints a public link to the app.

## Step 1: Install dependencies

In [ ]:
# prebuilt CUDA build of llama.cpp so Mistral runs on the GPU
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q gradio pymupdf easyocr llama-index-core llama-index-embeddings-huggingface huggingface_hub pandas

## Step 2: The pipeline (`rag_core.py`)

Ingestion, OCR, segmentation, indexing, routing, prompting and confidence. Written to a file so the UI and the evaluation can both import it.

In [ ]:
%%writefile rag_core.py
"""Core RAG pipeline for supplier PDFs (digital or scanned).

ingest:  PDF -> page text (PyMuPDF, or EasyOCR for scanned pages) -> page classification + document
         segmentation (Mistral, constrained JSON) -> chunks with metadata -> BGE-small vector index
answer:  question -> route to a document type (Mistral) -> filtered retrieval (with fallbacks)
         -> numbered-source prompt -> Mistral answer with [n] citations -> sources + confidence
"""
import json
import os
import re
import time
from functools import lru_cache

import fitz  # PyMuPDF
import llama_cpp
from llama_cpp import Llama
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

MODEL_PATH = os.environ.get("MISTRAL_GGUF", os.path.join(
    "/content/models" if os.path.isdir("/content") else "models",   # Colab, or a local models/ folder
    "mistral-7b-instruct-v0.2.Q4_K_M.gguf"))
TOP_K = 4
MIN_TEXT_CHARS = 50      # a page with less extractable text than this is treated as scanned
ROUTE_MIN_SCORE = 0.45   # if the best routed chunk scores below this, search everything instead

DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration",
             "Material Description", "Supplier Qualification", "Chain of Custody", "Other"]
TYPE_DEFINITIONS = """- Cover Letter: a letter or email addressed to the reader, e.g. about storage, shipping or what a package contains
- Certificate of Quality: certifies one product lot or batch (lot/batch numbers, manufacture/expiry dates, release tests), including third-party processing certificates
- Packaging Specification: packaging materials, configuration, packaging part numbers, drawings, packaging change history
- BSE/TSE Declaration: statement that products contain no animal-origin materials (BSE/TSE), animal origin statements
- Material Description: materials of construction, composition and physical properties of a product
- Supplier Qualification: supplier or vendor approval status, audits, certifications, sub-suppliers, quality agreement
- Chain of Custody: where assemblies are manufactured and how they move to the customer
- Other: none of the above"""

# ---------------------------------------------------------------- models (loaded once, lazily)
Settings.llm = None
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    query_instruction="Represent this sentence for searching relevant passages: ")

_llm, _ocr = None, None

def llm():
    global _llm
    if _llm is None:
        if not os.path.exists(MODEL_PATH):
            from huggingface_hub import hf_hub_download
            hf_hub_download("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
                            local_dir=os.path.dirname(MODEL_PATH))
        gpu = -1 if llama_cpp.llama_supports_gpu_offload() else 0
        _llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=gpu, seed=42, verbose=False)
    return _llm

def ocr_reader():
    global _ocr
    if _ocr is None:
        import easyocr
        import torch
        _ocr = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)
    return _ocr

@lru_cache(maxsize=None)
def ask_json(prompt, schema_json):
    # temperature 0 + JSON schema grammar: the answer is always valid JSON with allowed values only
    r = llm().create_chat_completion(messages=[{"role": "user", "content": prompt}],
                                     response_format={"type": "json_object", "schema": json.loads(schema_json)},
                                     temperature=0.0, max_tokens=80)
    return json.loads(r["choices"][0]["message"]["content"])

# ---------------------------------------------------------------- 1. text extraction (+ OCR)
def clean_ocr(text):
    """Fix the commonest OCR confusion in identifiers: letters O/o, l/I inside number-like tokens."""
    swap = str.maketrans({"O": "0", "o": "0", "l": "1", "I": "1"})
    def fix(m):
        tok = m.group(0)
        if sum(c.isdigit() for c in tok) / len(tok) >= 0.6:
            # keep a genuine letter prefix (the D in D000291847); fix confusable characters everywhere else
            keep = 1 if tok[0].isalpha() and tok[0] not in "OoIl" else 0
            return tok[:keep] + tok[keep:].translate(swap)
        return tok
    return re.sub(r"\b[A-Za-z0-9]{6,}\b", fix, text)

def deskew(gray):
    """Straighten a slightly rotated scan (rotation makes table rows drift across the page)."""
    import cv2
    import numpy as np
    ink = np.column_stack(np.where(gray < 128))[:, ::-1].astype(np.float32)   # (x, y) of dark pixels
    if len(ink) < 100:
        return gray
    angle = cv2.minAreaRect(ink)[-1]
    # OpenCV versions report this angle in [-90, 0) or (0, 90]; bring it into [-45, 45]
    if angle < -45:
        angle += 90
    elif angle > 45:
        angle -= 90
    if abs(angle) < 0.1 or abs(angle) > 10:               # already straight, or not a simple skew
        return gray
    h, w = gray.shape
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(gray, m, (w, h), flags=cv2.INTER_CUBIC, borderValue=255)

def ocr_text(page):
    """OCR a page and rebuild table rows: words whose boxes sit at the same height are joined into one line.
    Without this, every table cell comes out on its own line and a batch number loses its values."""
    import numpy as np
    pix = page.get_pixmap(dpi=200, colorspace=fitz.csGRAY)
    gray = deskew(np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width))
    results = ocr_reader().readtext(gray, detail=1)
    boxes = sorted(((sum(pt[1] for pt in b) / 4, min(pt[0] for pt in b), max(pt[1] for pt in b) - min(pt[1] for pt in b), t)
                    for b, t, _ in results), key=lambda x: x[0])
    rows, current = [], []
    for y, x, h, t in boxes:
        if current and abs(y - current[-1][0]) > 0.6 * max(h, current[-1][2]):
            rows.append(current)
            current = []
        current.append((y, x, h, t))
    if current:
        rows.append(current)
    return "\n".join(" | ".join(t for _, _, _, t in sorted(r, key=lambda c: c[1])) for r in rows)

def extract_pages(path):
    """One record per page; scanned pages (no text layer) are OCR'd."""
    pages = []
    with fitz.open(path) as pdf:
        for i, page in enumerate(pdf):
            text, ocr = page.get_text(), False
            if len(text.strip()) < MIN_TEXT_CHARS:
                text = clean_ocr(ocr_text(page))
                ocr = True
            pages.append({"source_file": os.path.basename(path), "page": i + 1, "text": text, "ocr": ocr})
    return pages

# ---------------------------------------------------------------- 2. classification + segmentation
PAGE_SCHEMA = json.dumps({"type": "object", "properties": {"doc_type": {"type": "string", "enum": DOC_TYPES},
                          "identifier": {"type": "string"}}, "required": ["doc_type", "identifier"]})

def describe_page(text):
    return ask_json(f"""Read this page from a bundle of pharmaceutical supplier documents.

Document types:
{TYPE_DEFINITIONS}

PAGE:
{text[:3000]}

Return JSON with:
"doc_type": the document type from the list above
"identifier": the document number, lot number or record code printed on this page that identifies this specific document. Copy it exactly as printed. Use "" if the page has none; do not make one up.""", PAGE_SCHEMA)

def verified_identifier(model_identifier, page_text):
    # keep only an ID-like token that really appears on the page (never a date)
    page = re.sub(r"\s+", " ", page_text)
    for token in re.findall(r"[A-Za-z0-9][A-Za-z0-9-]*", model_identifier):
        if re.match(r"^\d{4}-\d{2}-\d{2}$", token):
            continue
        for cand in [token] + token.split("-"):
            if len(cand) >= 5 and re.search(r"\d", cand) and cand in page:
                return cand
    return ""

def continues_previous(text):
    head = text[:400].lower()
    m = re.search(r"page\s+(\d+)\s*(?:of|/)\s*(\d+)", head)
    return "continued" in head or bool(m and int(m.group(1)) > 1)

def segment(pages):
    """Assign doc_type and doc_id to every page (rules from the Project 7 segmentation pipeline)."""
    prev, doc_n = None, 0
    for p in pages:
        info = describe_page(p["text"])
        ident = verified_identifier(info["identifier"], p["text"])
        cont = continues_previous(p["text"])
        if prev is None or prev["source_file"] != p["source_file"]:
            new = True
        elif cont:
            new = False
        elif info["doc_type"] != prev["doc_type"]:
            new = not (len(p["text"].split()) < 30 and not ident)   # short page without ID: keep with current doc
        else:
            new = bool(ident and prev["identifier"] and ident != prev["identifier"])
        if new:
            doc_n += 1
            p["doc_type"] = info["doc_type"]
        else:
            p["doc_type"] = prev["doc_type"]
        p["doc_id"] = f"{p['source_file']}#{doc_n}"
        p["identifier"] = ident or (prev["identifier"] if prev and not new else "")
        prev = p
    # page range of each document, stored on every page of it
    ranges = {}
    for p in pages:
        ranges.setdefault(p["doc_id"], []).append(p["page"])
    for p in pages:
        r = ranges[p["doc_id"]]
        p["page_range"] = f"{r[0]}" if len(r) == 1 else f"{r[0]}-{r[-1]}"
    return pages

# ---------------------------------------------------------------- 3. chunking + index
def build_index(pages):
    docs = []
    for p in pages:
        meta = {k: p[k] for k in ["source_file", "page", "page_range", "doc_id", "doc_type", "identifier", "ocr"]}
        docs.append(Document(text=p["text"], metadata=meta,
                             excluded_embed_metadata_keys=[k for k in meta if k != "doc_type"]))
    # split at line breaks, so a table row (one line) is never cut in half between two chunks
    nodes = SentenceSplitter(chunk_size=256, chunk_overlap=30, paragraph_separator="\n").get_nodes_from_documents(docs)
    for i, n in enumerate(nodes):
        n.metadata["chunk_index"] = i
        n.excluded_embed_metadata_keys = n.excluded_embed_metadata_keys + ["chunk_index"]
    return VectorStoreIndex(nodes), len(nodes)

CACHE_DIR = os.environ.get("RAG_CACHE", ".rag_cache")

def cache_key(paths):
    # same files in the same order -> same key, whatever their names or upload location
    import hashlib
    h = hashlib.sha256()
    for path in paths:
        with open(path, "rb") as f:
            h.update(hashlib.sha256(f.read()).digest())
    return h.hexdigest()[:16]

def ingest(paths, progress=None):
    """Full ingestion. Returns (index, pages, stats).
    The slow steps (OCR and page classification) are cached on disk, so the same files load in seconds next time."""
    t0 = time.perf_counter()
    paths = sorted(paths, key=os.path.basename)   # upload order doesn't matter
    cache_file = os.path.join(CACHE_DIR, cache_key(paths) + ".json")
    if os.path.exists(cache_file):
        if progress: progress(0.5, desc="Loading processed pages from cache")
        with open(cache_file, encoding="utf-8") as f:
            pages = json.load(f)
        t_extract = t_segment = 0.0
        cached = True
    else:
        pages, cached = [], False
        for i, path in enumerate(paths):
            if progress: progress(i / max(len(paths), 1) * 0.5, desc=f"Reading {os.path.basename(path)}")
            pages += extract_pages(path)
        t_extract = time.perf_counter() - t0
        if progress: progress(0.5, desc="Classifying pages and finding document boundaries")
        segment(pages)
        t_segment = time.perf_counter() - t0 - t_extract
        os.makedirs(CACHE_DIR, exist_ok=True)
        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(pages, f)
    if progress: progress(0.9, desc="Embedding and indexing")
    index, n_chunks = build_index(pages)
    stats = {"files": len(paths), "pages": len(pages), "ocr_pages": sum(p["ocr"] for p in pages),
             "documents": len({p["doc_id"] for p in pages}), "chunks": n_chunks,
             "extract_s": round(t_extract, 1), "segment_s": round(t_segment, 1),
             "total_s": round(time.perf_counter() - t0, 1), "cached": cached}
    return index, pages, stats

# ---------------------------------------------------------------- 4. routing + retrieval
QUERY_SCHEMA = json.dumps({"type": "object", "properties": {"doc_type": {"type": "string", "enum": DOC_TYPES}},
                           "required": ["doc_type"]})

def route_query(question):
    t = ask_json(f"""Which type of document in a pharmaceutical supplier bundle is most likely to answer this question?

Document types:
{TYPE_DEFINITIONS}

QUESTION: {question}

Return JSON with "doc_type". Use "Other" if the question does not point to one type.""", QUERY_SCHEMA)["doc_type"]
    return None if t == "Other" else t

def identifiers_in(question):
    return re.findall(r"\b(?:[A-Z]{1,}-?[A-Z0-9-]*\d[A-Z0-9-]*|\d{6,})\b", question)

def exact_matches(index, question, ids):
    """Chunks containing an ID from the question, ranked by their embedding similarity to the question."""
    from llama_index.core.schema import NodeWithScore
    hits = [n for n in index.docstore.docs.values() if any(i in n.get_content() for i in ids)]
    if not hits:
        return []
    q = Settings.embed_model.get_query_embedding(question)
    def sim(n):
        e = Settings.embed_model.get_text_embedding(n.get_content(metadata_mode="embed"))
        return sum(a * b for a, b in zip(q, e)) / ((sum(a * a for a in q) ** 0.5) * (sum(b * b for b in e) ** 0.5))
    return sorted((NodeWithScore(node=n, score=sim(n)) for n in hits), key=lambda x: -x.score)[:TOP_K]

def retrieve(index, question, mode="Auto"):
    """mode: 'Auto' (route by document type), 'All documents', or a specific document type.
    Returns (nodes, route_info)."""
    search = lambda dt: index.as_retriever(similarity_top_k=TOP_K, filters=None if dt is None else
                                           MetadataFilters(filters=[MetadataFilter(key="doc_type", value=dt)])
                                           ).retrieve(question)
    if mode == "All documents":
        return search(None), {"doc_type": None, "note": "searched all documents"}
    routed = route_query(question) if mode == "Auto" else mode
    if routed is None:
        return search(None), {"doc_type": None, "note": "question not specific to one document type; searched all"}
    nodes = search(routed)
    text = " ".join(n.get_content() for n in nodes)
    ids = [i for i in identifiers_in(question) if len(i) >= 5]
    if mode == "Auto" and ids and not any(i in text for i in ids):
        # exact-match lookup: chunks that literally contain the ID come first, then the best unfiltered matches
        exact = exact_matches(index, question, ids)
        rest = [n for n in search(None) if n.node.node_id not in {e.node.node_id for e in exact}]
        note = f"routed to {routed}, but {ids[0]} was not there; " + (
            f"found it by exact match" if exact else "searched all")
        return (exact + rest)[:TOP_K], {"doc_type": routed, "note": note}
    if mode == "Auto" and (not nodes or nodes[0].score < ROUTE_MIN_SCORE):
        return search(None), {"doc_type": routed, "note": f"weak match in {routed}; searched all"}
    return nodes, {"doc_type": routed, "note": f"routed to {routed}"}

# ---------------------------------------------------------------- 5. prompt + answer
def build_prompt(question, nodes):
    sources = "\n\n".join(
        f"[{i}] {n.metadata['doc_type']}, {n.metadata['source_file']}, page {n.metadata['page']}\n{n.get_content()}"
        for i, n in enumerate(nodes, 1))
    return f"""You are a pharmaceutical quality assistant. Answer the question using ONLY the numbered sources below.

Rules:
- After every fact, cite the source number in square brackets, e.g. [1] or [2].
- Copy numbers, dates, lot numbers and units exactly as written in the sources.
- Tables are written one row per line with cells separated by "|"; the first row of a table names the columns. Read a value from the same row as the item it belongs to.
- If the sources do not contain the answer, reply exactly: "I could not find this in the uploaded documents."
- Answer only the question asked. Do not comment on sources that are not relevant, and do not add further questions.
- Be concise.

SOURCES:
{sources}

QUESTION: {question}
ANSWER:"""

def stream_answer(prompt):
    for chunk in llm().create_chat_completion(messages=[{"role": "user", "content": prompt}],
                                              temperature=0.0, max_tokens=350, stream=True,
                                              stop=["QUESTION:", "\nQ:", "SOURCES:"]):
        yield chunk["choices"][0]["delta"].get("content", "")

NOT_FOUND = "could not find this in the uploaded documents"

DATE = r"\b\d{4}[-/.]\d{1,2}[-/.]\d{1,2}\b"

def numbers_in(text):
    text = re.sub(r"\[\d+\]", " ", text)
    # a reformatted date (2021-07-12 for 20210712) counts as its digits joined, not as three numbers
    dates = {re.sub(r"\D", "", d) for d in re.findall(DATE, text)}
    return set(re.findall(r"\d+(?:\.\d+)?", re.sub(DATE, " ", text))) | dates

def assess(answer, nodes):
    """Confidence from two checks that can be computed, not guessed:
    retrieval similarity of the best chunk, and whether every number in the answer appears in the sources."""
    cited = sorted({int(c) for c in re.findall(r"\[(\d+)\]", answer) if 1 <= int(c) <= len(nodes)})
    source_text = " ".join(n.get_content() for n in nodes)
    src_numbers = numbers_in(source_text) | numbers_in(source_text.replace(" ", "")) | set(re.findall(r"\d+", source_text))
    ans_numbers = numbers_in(answer)
    unsupported = sorted(ans_numbers - src_numbers)
    grounding = 1.0 if not ans_numbers else 1 - len(unsupported) / len(ans_numbers)
    top = nodes[0].score if nodes else 0.0
    if "could not find" in answer.lower():
        level = "Low"   # a full or partial "not found" answer
    elif top >= 0.60 and grounding == 1.0 and cited:
        level = "High"
    elif top >= 0.50 and grounding >= 0.8:
        level = "Medium"
    else:
        level = "Low"
    return {"level": level, "top_similarity": round(top, 3), "grounding": round(grounding, 2),
            "unsupported_numbers": unsupported, "cited": cited, "chunks": len(nodes)}

## Step 3: The Gradio UI (`app.py`)

In [ ]:
%%writefile app.py
"""Pharma Supplier Document Assistant: Gradio UI for the full RAG pipeline in rag_core.py."""
import os
import time
from datetime import datetime

import gradio as gr
import pandas as pd

import rag_core as rc

SCOPES = ["Auto (route by document type)", "All documents"] + [t for t in rc.DOC_TYPES if t != "Other"]
BADGE = {"High": "🟢 High", "Medium": "🟡 Medium", "Low": "🔴 Low"}
EMPTY_SOURCES = pd.DataFrame(columns=["#", "Cited", "Document type", "File", "Page", "Similarity"])

# ---------------------------------------------------------------- callbacks
def process(files, progress=gr.Progress()):
    if not files:
        return None, "⚠️ Upload at least one PDF first.", pd.DataFrame()
    paths = [f if isinstance(f, str) else f.name for f in files]
    index, pages, s = rc.ingest(paths, progress=progress)
    library = (pd.DataFrame(pages)
               .groupby("doc_id", sort=False)
               .agg(Type=("doc_type", "first"), Pages=("page_range", "first"), File=("source_file", "first"),
                    ID=("identifier", "first"), OCR=("ocr", "any"))
               .reset_index(drop=True))
    library["OCR"] = library["OCR"].map({True: "scanned (OCR)", False: "digital"})
    status = (f"✅ **{s['files']} file(s) · {s['pages']} pages · {s['documents']} documents · {s['chunks']} chunks** "
              f"({s['ocr_pages']} pages OCR'd) in {s['total_s']:.0f}s" + (" · loaded from cache" if s["cached"] else ""))
    return index, status, library

def ask(question, history, index, scope):
    question = (question or "").strip()
    if not question:
        yield history, "", gr.update(), EMPTY_SOURCES
        return
    history = history + [{"role": "user", "content": question}]
    if index is None:
        history.append({"role": "assistant", "content": "Please upload PDFs and click **Process documents** first."})
        yield history, "", "⚠️ No documents indexed yet.", EMPTY_SOURCES
        return

    history.append({"role": "assistant", "content": "⏳ Finding the right document..."})
    yield history, "", "🔎 Routing and retrieving...", EMPTY_SOURCES
    t0 = time.perf_counter()
    mode = "Auto" if scope.startswith("Auto") else scope
    nodes, route = rc.retrieve(index, question, mode)
    t_retrieve = time.perf_counter() - t0

    sources = pd.DataFrame([{"#": i, "Cited": "", "Document type": n.metadata["doc_type"],
                             "File": n.metadata["source_file"], "Page": n.metadata["page"],
                             "Similarity": f"{n.score:.0%}"} for i, n in enumerate(nodes, 1)])
    history[-1]["content"] = "✍️ Writing the answer..."
    yield history, "", f"✍️ {route['note'].capitalize()}. Writing the answer from {len(nodes)} chunks...", sources

    answer = ""
    for piece in rc.stream_answer(rc.build_prompt(question, nodes)):
        answer += piece
        history[-1]["content"] = answer + " ▌"
        yield history, "", gr.update(), sources

    a = rc.assess(answer, nodes)
    sources["Cited"] = ["✔" if i in a["cited"] else "" for i in sources["#"]]
    cited_refs = " · ".join(f"[{i}] {nodes[i-1].metadata['source_file']} p.{nodes[i-1].metadata['page']}" for i in a["cited"])
    history[-1]["content"] = (answer.strip() + f"\n\n**Confidence: {BADGE[a['level']]}** · {a['chunks']} chunks"
                              + (f" · 📎 {cited_refs}" if cited_refs else ""))
    warn = f" · ⚠️ numbers not found in sources: {', '.join(a['unsupported_numbers'])}" if a["unsupported_numbers"] else ""
    details = (f"**Confidence {BADGE[a['level']]}** · best match {a['top_similarity']:.0%} · "
               f"numbers grounded {a['grounding']:.0%} · {a['chunks']} chunks · {route['note']} · "
               f"retrieval {t_retrieve:.1f}s, total {time.perf_counter() - t0:.0f}s{warn}")
    yield history, "", details, sources

def export_chat(history):
    if not history:
        return None, "Nothing to save yet."
    path = os.path.abspath(f"chat_{datetime.now():%Y%m%d_%H%M%S}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# Supplier document chat, {datetime.now():%d %B %Y %H:%M}\n\n")
        for m in history:
            f.write(f"**{'You' if m['role'] == 'user' else 'Assistant'}:** {m['content']}\n\n")
    return path, f"💾 Saved {len(history) // 2} question(s)."

# ---------------------------------------------------------------- layout
CSS = """
#hero h1 {margin-bottom: 0}
#hero p {color: #64748b; margin-top: 4px}
#details {min-height: 2.6em; font-size: 0.95em}
"""
THEME = gr.themes.Soft(primary_hue="indigo", secondary_hue="sky", neutral_hue="slate")

with gr.Blocks(title="Pharma Supplier Document Assistant") as demo:
    index_state = gr.State(None)
    gr.Markdown("# 💊 Pharma Supplier Document Assistant\n"
                "Upload supplier PDFs (digital or scanned). Every answer cites its sources and shows how confident it is.",
                elem_id="hero")
    with gr.Row():
        with gr.Column(scale=4):
            files = gr.File(label="📄 Supplier PDFs", file_types=[".pdf"], file_count="multiple")
            process_btn = gr.Button("⚙️ Process documents", variant="primary")
            status = gr.Markdown("Upload PDFs to begin. Scanned pages are read with OCR.")
            library = gr.Dataframe(label="📚 Document library (detected automatically)", interactive=False, wrap=True)
            scope = gr.Dropdown(SCOPES, value=SCOPES[0], label="🔍 Search scope")
            with gr.Row():
                save_btn = gr.Button("💾 Save chat")
                clear_btn = gr.Button("🗑️ Clear")
            saved = gr.File(label="Chat export", interactive=False, visible=True)
        with gr.Column(scale=7):
            chatbot = gr.Chatbot(label="Assistant", height=470,
                                 placeholder="Ask about storage conditions, lot expiry dates, sterilization doses, audits...")
            with gr.Row():
                question = gr.Textbox(placeholder="Ask a question about your documents...", show_label=False, scale=6)
                send = gr.Button("Ask", variant="primary", scale=1)
            details = gr.Markdown("", elem_id="details")
            sources = gr.Dataframe(EMPTY_SOURCES, label="📎 Sources for the last answer", interactive=False, wrap=True)
            gr.Examples(["What is the recommended storage temperature for the AKTA ready flow kits?",
                         "What gamma dose range did batch D000291847 receive?",
                         "When does the BSE/TSE risk assessment expire?",
                         "Which sub-suppliers provide materials for the West stoppers?"], inputs=question)

    process_btn.click(process, inputs=files, outputs=[index_state, status, library])
    for trigger in (send.click, question.submit):
        trigger(ask, inputs=[question, chatbot, index_state, scope], outputs=[chatbot, question, details, sources])
    save_btn.click(export_chat, inputs=chatbot, outputs=[saved, details])
    clear_btn.click(lambda: ([], "", EMPTY_SOURCES), outputs=[chatbot, details, sources])

if __name__ == "__main__":
    demo.launch(theme=THEME, css=CSS)

## Step 4: Test documents

The two course sample bundles. The second one is also turned into a **scanned copy** (pages rasterised, slightly rotated, with noise and no text layer) so the OCR path can be tested and measured against the original.

In [ ]:
import io, os, random, urllib.request
import fitz, numpy as np
from PIL import Image, ImageFilter

os.makedirs("pdfs", exist_ok=True)
for name, drive_id in {"pharma-blob-sample.pdf": "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw",
                       "pharma-blob-test.pdf": "1atOj_CzvPRm-CskbhvJdzlJIpcEanVu1"}.items():
    if not os.path.exists(f"pdfs/{name}"):
        urllib.request.urlretrieve(f"https://drive.google.com/uc?export=download&id={drive_id}", f"pdfs/{name}")

# simulate a scan: 150 dpi greyscale image, slight rotation, noise, JPEG compression, no text layer
random.seed(7)
src, scan = fitz.open("pdfs/pharma-blob-test.pdf"), fitz.open()
for page in src:
    pix = page.get_pixmap(dpi=150, colorspace=fitz.csGRAY)
    img = Image.frombytes("L", (pix.width, pix.height), pix.samples)
    img = img.rotate(random.uniform(-1.2, 1.2), fillcolor=255, resample=Image.BICUBIC)
    arr = np.clip(np.array(img).astype(np.int16) + np.random.default_rng(7).normal(0, 12, img.size[::-1]), 0, 255)
    img = Image.fromarray(arr.astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.4))
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=70)
    new = scan.new_page(width=page.rect.width, height=page.rect.height)
    new.insert_image(new.rect, stream=buf.getvalue())
scan.save("pdfs/pharma-blob-test-scanned.pdf")
print("text layer on scanned page 1:", len(fitz.open("pdfs/pharma-blob-test-scanned.pdf")[0].get_text().strip()), "characters")

## Step 5 (optional): Evaluation

Ten questions with known answers across the digital Cytiva bundle and the scanned West bundle, including one out-of-scope question. Results from my run (laptop CPU):

- **Answers correct:** 7/10
- **OCR word recall:** 99.5%
- **Retrieval with routing (top-4):** Recall@4 94.4%, MRR 0.89, Precision@4 50.0%, hit rate 100%
- **Searching everything instead:** Recall@4 83.3%, MRR 0.72, hit rate 88.9%
- **Citations to a correct page:** 8/9

Running it re-ingests both PDFs (a few minutes on a T4).

In [ ]:
%%writefile evaluate.py
"""Evaluate the full RAG pipeline on questions with known answers.

usage: python evaluate.py <digital pdf> <scanned pdf> <digital original of the scanned pdf> <output json>
"""
import json
import re
import sys
import time

import fitz

import rag_core as rc

digital_pdf, scanned_pdf, scanned_original, out_path = sys.argv[1:5]
SAMPLE, SCANNED = "pharma-blob-sample.pdf", scanned_pdf.replace("\\", "/").split("/")[-1]

def pages_of(where):
    return [] if where is None else (where if isinstance(where, list) else [where])

def digits(t):
    return re.sub(r"[-/. ]", "", t.lower())

# question, strings the answer must contain, (file, page) or list of them that hold the answer
TESTS = [
    ("What is the recommended storage temperature for the AKTA ready flow kits?", ["+5"], (SAMPLE, 1)),
    ("When does the BSE/TSE risk assessment for the AKTA flow kits expire?", ["15 March 2028"], (SAMPLE, 6)),
    ("What is the expiration date of lot 15102934?", ["20210712"], (SAMPLE, 3)),
    ("When was Cytiva last audited?", ["20 November 2023"], (SAMPLE, 8)),
    ("What gamma dose range did batch D000291847 receive?", ["27.3", "38.1"], (SCANNED, 10)),
    ("Until when is the West animal origin statement valid?", ["2029"], (SCANNED, 6)),
    # "re:" = regular expression; "not:" = must NOT appear (stricter checks after reading the first run's answers)
    ("Which sub-suppliers provide materials for the West stoppers?",
     [r"re:ExxonMobil[^.;]{0,40}chlorobutyl", r"re:Daikin[^.;]{0,40}PTFE"], (SCANNED, 9)),
    ("What change was made under ECN-2024-0027?", ["28", "32", "ECT", "not:minute"], (SCANNED, 5)),
    ("What is the Shore A hardness specification of the West stopper?", ["50"], [(SCANNED, 4), (SCANNED, 7)]),
    ("What is the capital of France?", [rc.NOT_FOUND], None),
]

t0 = time.perf_counter()
index, pages, stats = rc.ingest([digital_pdf, scanned_pdf])
print("ingest:", json.dumps(stats), flush=True)

# OCR quality: word recall of the OCR'd pages against the digital original
def words(t):
    return re.findall(r"[a-z0-9]+", t.lower())
with fitz.open(scanned_original) as pdf:
    truth = [words(p.get_text()) for p in pdf]
ocr_pages = [p for p in pages if p["ocr"]]
recalls = []
for p, t in zip(ocr_pages, truth):
    got = set(words(p["text"]))
    recalls.append(sum(w in got for w in t) / len(t))
ocr_recall = sum(recalls) / len(recalls)
print(f"OCR word recall: {ocr_recall:.1%} (min page {min(recalls):.1%})", flush=True)

# retrieval metrics: auto-routing vs searching everything (no generation needed)
# a retrieved chunk is relevant if it comes from a page that holds the answer
retrieval = []
for q, _, where in TESTS:
    if where is None:
        continue
    row = {"question": q}
    relevant_pages = pages_of(where)
    for mode in ["All documents", "Auto"]:
        t1 = time.perf_counter()
        nodes, route = rc.retrieve(index, q, mode)   # Auto includes the (uncached) routing LLM call
        latency = time.perf_counter() - t1
        found = [(n.metadata["source_file"], n.metadata["page"]) for n in nodes]
        ranks = [r for r, p in enumerate(found, 1) if p in relevant_pages]
        row[mode] = bool(ranks)
        row[mode + " metrics"] = {
            "recall": len(set(found) & set(relevant_pages)) / len(relevant_pages),
            "precision": len(ranks) / len(nodes),
            "rr": 1 / ranks[0] if ranks else 0.0,
            "latency_s": round(latency, 2)}
        if mode == "Auto":
            row["route"] = route["note"]
    retrieval.append(row)
    print("retrieval:", row, flush=True)

def mean(xs):
    return sum(xs) / len(xs)
retrieval_summary = {mode: {m: round(mean([r[mode + " metrics"][m] for r in retrieval]), 3)
                            for m in ["recall", "precision", "rr", "latency_s"]} | {"hit_rate": mean([r[mode] for r in retrieval])}
                     for mode in ["All documents", "Auto"]}
print("retrieval summary:", retrieval_summary, flush=True)

# full answers
results = []
for q, must, where in TESTS:
    t1 = time.perf_counter()
    nodes, route = rc.retrieve(index, q, "Auto")   # routing answer is cached from the retrieval pass above
    t_retrieve = time.perf_counter() - t1
    t2 = time.perf_counter()
    answer = "".join(rc.stream_answer(rc.build_prompt(q, nodes))).strip()
    t_generate = time.perf_counter() - t2
    t_total = time.perf_counter() - t1
    a = rc.assess(answer, nodes)
    def check(m):
        if m.startswith("re:"):
            return bool(re.search(m[3:], answer, re.I))
        if m.startswith("not:"):
            return m[4:].lower() not in answer.lower()
        return m.lower() in answer.lower() or digits(m) in digits(answer)
    correct = all(check(m) for m in must)
    cited_right = None if where is None else any(
        (nodes[c - 1].metadata["source_file"], nodes[c - 1].metadata["page"]) in pages_of(where) for c in a["cited"])
    results.append({"question": q, "answer": answer, "correct": correct, "cited_right_page": cited_right,
                    "route": route["note"], **a, "retrieve_s": round(t_retrieve, 2), "generate_s": round(t_generate, 1),
                    "total_s": round(t_total, 1),
                    "sources": [f"{n.metadata['source_file']} p{n.metadata['page']} ({n.score:.2f})" for n in nodes]})
    print(json.dumps(results[-1]), flush=True)

json.dump({"ingest": stats, "ocr_word_recall": ocr_recall, "ocr_page_recalls": recalls,
           "retrieval": retrieval, "retrieval_summary": retrieval_summary, "answers": results},
          open(out_path, "w"), indent=2)
print("done in", round(time.perf_counter() - t0), "s")

In [ ]:
!python evaluate.py pdfs/pharma-blob-sample.pdf pdfs/pharma-blob-test-scanned.pdf pdfs/pharma-blob-test.pdf eval_results.json 2>/dev/null | grep -v '^{'

## Step 6: Launch the app

Upload `pdfs/pharma-blob-sample.pdf` and `pdfs/pharma-blob-test-scanned.pdf` (download them from the file browser on the left), click **Process documents**, then ask a question. The public `gradio.live` link works while this session is running.

In [ ]:
import app
app.demo.launch(share=True, theme=app.THEME, css=app.CSS)